In [4]:
import torch
import torch.nn.functional as F

from src.config import cfg
from src.tokenizer import Tokenizer

In [5]:
tokenizer = Tokenizer.load(cfg.paths.tokenizer)

ckpt = torch.load(
    f"{cfg.paths.checkpoint_dir}/latest.pt",
    map_location="cpu",
    weights_only=False,
)

state = ckpt["model"]

u = state["u_embeddings.weight"]
v = state["v_embeddings.weight"]

emb = (u + v) / 2
emb = F.normalize(emb, dim=1)

In [6]:
def most_similar(word, topk=10):
    idx = tokenizer.index(word)

    sims = emb @ emb[idx]
    sims[idx] = -1e9

    ids = torch.topk(sims, topk).indices.tolist()

    return [(tokenizer.word(i), float(sims[i])) for i in ids]

In [10]:
# a - b + c
def analogy(a, b, c, topk=10):
    a_idx = tokenizer.index(a)
    b_idx = tokenizer.index(b)
    c_idx = tokenizer.index(c)

    query = emb[a_idx] - emb[b_idx] + emb[c_idx]
    query = F.normalize(query, dim=0)

    sims = emb @ query
    sims[[a_idx, b_idx, c_idx]] = -1e9

    ids = torch.topk(sims, topk).indices.tolist()

    return [(tokenizer.word(i), float(sims[i])) for i in ids]

In [12]:
analogies = [

    {
        "category": "cinsiyet",
        "expected": "kraliçe",
        "query": ("kral", "erkek", "kadın"),
    },
    {
        "category": "cinsiyet",
        "expected": "anne",
        "query": ("baba", "erkek", "kadın"),
    },
    {
        "category": "cinsiyet",
        "expected": "teyze",
        "query": ("amca", "erkek", "kadın"),
    },
    {
        "category": "cinsiyet",
        "expected": "kadın",
        "query": ("adam", "erkek", "kadın"),
    },
    {
        "category": "akrabalık",
        "expected": "kız",
        "query": ("oğul", "erkek", "kadın"),
    },

    {
        "category": "başkent",
        "expected": "paris",
        "query": ("ankara", "türkiye", "fransa"),
    },
    {
        "category": "başkent",
        "expected": "paris",
        "query": ("berlin", "almanya", "fransa"),
    },
    {
        "category": "başkent",
        "expected": "berlin",
        "query": ("londra", "ingiltere", "almanya"),
    },
    {
        "category": "başkent",
        "expected": "tokyo",
        "query": ("moskova", "rusya", "japonya"),
    },
    {
        "category": "başkent",
        "expected": "roma",
        "query": ("paris", "fransa", "italya"),
    },

    {
        "category": "şehir-ülke",
        "expected": "paris",
        "query": ("istanbul", "türkiye", "fransa"),
    },
    {
        "category": "şehir-ülke",
        "expected": "tokyo",
        "query": ("berlin", "almanya", "japonya"),
    },

    {
        "category": "fiil-zaman",
        "expected": "geldi",
        "query": ("gitmek", "gitti", "gelmek"),
    },
    {
        "category": "fiil-zaman",
        "expected": "okudu",
        "query": ("yazmak", "yazdı", "okumak"),
    },
    {
        "category": "fiil-zaman",
        "expected": "yürüdü",
        "query": ("koşmak", "koştu", "yürümek"),
    },

    {
        "category": "meslek-mekan",
        "expected": "öğretmen",
        "query": ("doktor", "hastane", "okul"),
    },
    {
        "category": "meslek-mekan",
        "expected": "doktor",
        "query": ("öğretmen", "okul", "hastane"),
    },
    {
        "category": "araç",
        "expected": "kaptan",
        "query": ("pilot", "uçak", "gemi"),
    },

    {
        "category": "ülke-yönetim",
        "expected": "başkan",
        "query": ("cumhurbaşkanı", "türkiye", "amerika"),
    },

    {
        "category": "akrabalık",
        "expected": "baba",
        "query": ("anne", "kadın", "erkek"),
    },
    {
        "category": "akrabalık",
        "expected": "abi",
        "query": ("abla", "kadın", "erkek"),
    },

    {
        "category": "kıta-ülke",
        "expected": "avrupa",
        "query": ("asya", "çin", "fransa"),
    },

    {
        "category": "spor",
        "expected": "raket",
        "query": ("futbol", "top", "tenis"),
    },
    {
        "category": "spor",
        "expected": "kale",
        "query": ("basketbol", "pota", "futbol"),
    },

    {
        "category": "teknoloji",
        "expected": "sql",
        "query": ("python", "programlama", "veritabanı"),
    },
]

In [13]:
for item in analogies:

    a, b, c = item["query"]

    print(f"\n[{item['category']}]")
    print(f"Beklenen: {item['expected']}")
    print(f"{a} - {b} + {c}")

    print(analogy(a, b, c))


[cinsiyet]
Beklenen: kraliçe
kral - erkek + kadın
[('kralın', 0.5725920796394348), ('krala', 0.5447142124176025), ('kralı', 0.5419153571128845), ('aeryse', 0.5401062965393066), ('kraldan', 0.5309479236602783), ('wormsta', 0.523047924041748), ('kraliçedir', 0.5225113034248352), ('elizabethden', 0.5197600722312927), ('birendra', 0.518351674079895), ('veliahtının', 0.5159949064254761)]

[cinsiyet]
Beklenen: anne
baba - erkek + kadın
[('babanın', 0.5823487043380737), ('babayla', 0.5264649391174316), ('babayı', 0.5264482498168945), ('babaanne', 0.51650071144104), ('mardinli', 0.508500337600708), ('abdüssamed', 0.5080295205116272), ('manuş', 0.5064501762390137), ('yaga', 0.506176233291626), ('macide', 0.5049865245819092), ('eftalya', 0.5047745704650879)]

[cinsiyet]
Beklenen: teyze
amca - erkek + kadın
[('ayşeyi', 0.6170535087585449), ('amcanın', 0.6138120293617249), ('meyhaneye', 0.6121962070465088), ('pusat', 0.6117353439331055), ('leylayı', 0.6115589141845703), ('amcaya', 0.6113756895065